# Integrity diagnostics

### Setup libraries and pandas view:

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/chembl_glp1.csv", sep=';')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

### Filtering data based on previous audit:

In [81]:
conclusive_rows = df[(df['Standard Type'] == 'Potency') 
                     & (df['Standard Value'] != 28183.8) 
                     & (df['Comment'] != 'inconclusive')
                     & (df['Assay ChEMBL ID'] == 'CHEMBL2114788')].dropna(axis=1, how='all')

print(conclusive_rows.shape)

(21412, 28)


### Zero-variance / constant column audit:

In [82]:
print(f"There are {(conclusive_rows.nunique() == 1).sum()} columns containing a single unique value:")

constant_cols = conclusive_rows.columns[conclusive_rows.nunique() == 1]

for idx, val in conclusive_rows[constant_cols].iloc[0].items():
    print(f"{idx} = {val}")

df_modelling_ready = conclusive_rows.drop(columns=constant_cols)
df_modelling_ready.shape

conclusive_rows[constant_cols].isnull().sum() # Checking for nulls

There are 17 columns containing a single unique value:
Standard Type = Potency
Standard Units = nM
Uo Units = UO_0000065
Potential Duplicate = 0
Assay ChEMBL ID = CHEMBL2114788
Assay Description = PubChem BioAssay. qHTS of GLP-1 Receptor Inverse Agonists (Inhibition Mode). (Class of assay: confirmatory) 
Assay Type = F
BAO Format ID = BAO_0000019
BAO Label = assay format
Assay Organism = Homo sapiens
Target ChEMBL ID = CHEMBL1784
Target Name = Glucagon-like peptide 1 receptor
Target Organism = Homo sapiens
Target Type = SINGLE PROTEIN
Document ChEMBL ID = CHEMBL1201862
Source ID = 7
Source Description = PubChem BioAssays


Standard Type          0
Standard Units         0
Uo Units               0
Potential Duplicate    0
Assay ChEMBL ID        0
Assay Description      0
Assay Type             0
BAO Format ID          0
BAO Label              0
Assay Organism         0
Target ChEMBL ID       0
Target Name            0
Target Organism        0
Target Type            0
Document ChEMBL ID     0
Source ID              0
Source Description     0
dtype: int64

### Derived/redundant column audit:

In [83]:
print(conclusive_rows.head(1))
print(conclusive_rows.info())

    Molecule ChEMBL ID Molecule Name  Molecule Max Phase  Molecular Weight  #RO5 Violations  AlogP Compound Key                             Smiles Standard Type  Standard Value Standard Units   Comment    Uo Units  Potential Duplicate Assay ChEMBL ID                                  Assay Description Assay Type BAO Format ID     BAO Label Assay Organism Target ChEMBL ID                       Target Name Target Organism     Target Type Document ChEMBL ID  Source ID Source Description   Value
129      CHEMBL1537791           NaN                 NaN            321.41              0.0   3.61  SID24813643  CSc1ccc(CNc2nc3ccccc3n3cnnc23)cc1       Potency          5623.4             nM  inactive  UO_0000065                    0   CHEMBL2114788  PubChem BioAssay. qHTS of GLP-1 Receptor Inver...          F   BAO_0000019  assay format   Homo sapiens       CHEMBL1784  Glucagon-like peptide 1 receptor    Homo sapiens  SINGLE PROTEIN      CHEMBL1201862          7  PubChem BioAssays  5.6234
<class '

### Schema verification checks:

In [84]:
expected_cols = ['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Value', 'Standard Units', 'Comment', 'Uo Units', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type', 'Document ChEMBL ID', 'Source ID', 'Source Description', 'Value']

def check_columns(df, expected_cols):
    missing = set(expected_cols) - set(df.columns)
    extra = set(df.columns) - set(expected_cols)
    if not missing and not extra:
        print("Columns match expected schema")
    else:
        if missing:
            print(f"Missing columns: {missing}")
        if extra:
            print(f"Unexpected columns: {extra}")

def check_single_assay(df, col='Assay ChEMBL ID'):
    n = df[col].nunique()
    if n == 1:
        print(f"Single {col}: {df[col].iloc[0]}")
    else:
        print(f"Multiple values ({df[col].unique()}) in {col}")

def check_no_placeholder_value(df, col='Standard Value', placeholder=28183.8):
    n = (df[col] == placeholder).sum()
    if n == 0:
        print(f"No {col} rows at placeholder: {placeholder}")
    else:
        print(f"{n} rows in {col} at placeholder value {placeholder}")

def check_value_range(df, col='Standard Value', min_val=0):
    n = (df[col] <= min_val).sum()
    if n == 0:
        print(f"All {col} values > {min_val}")
    else:
        print(f"{n} rows found with {col} values <= {min_val}")

def check_allowed_categories(df, col, allowed):
    unexpected = set(df[col].dropna().unique()) - set(allowed)
    if not unexpected:
        print(f"{col} contains only allowed categories")
    else:
        print(f"Unexpected values in {col}: {unexpected}")

print(f"Running schema checks:")
check_columns(conclusive_rows, expected_cols)
check_single_assay(conclusive_rows)
check_value_range(conclusive_rows)
check_no_placeholder_value(conclusive_rows)
check_allowed_categories(conclusive_rows, 'Comment', ['active', 'inactive'])

Running schema checks:
Columns match expected schema
Single Assay ChEMBL ID: CHEMBL2114788
All Standard Value values > 0
No Standard Value rows at placeholder: 28183.8
Comment contains only allowed categories
